# Kuru active blocks — UTC window

Count **unique blocks** where any Kuru contract was touched in a UTC day, with **per-market / per-group activity** breakdown.

A block counts if:
- a **top-level transaction** has `to` in Kuru addresses, **or**
- an **internal trace** has `to` in Kuru addresses (router to market, etc.)

Markets are grouped from `protocols/kuru_addresses.json` (e.g. `MON/USDC` = market + operator + backrun + fastlane addresses).

**Data sources**
- Addresses: `protocols/kuru_addresses.json` (from `kuru_address.txt`)
- Block range: private RPC from environment
- Activity: private server/indexer source from environment
- Auth: private environment variables

In [3]:
%pip install hypersync python-dotenv tqdm requests -q

import asyncio
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import hypersync
import requests
from dotenv import load_dotenv
from tqdm.auto import tqdm

# ========== CONFIGURATION ==========
START_UTC = "2026-06-25T00:00:00.000Z"
END_UTC = "2026-06-25T23:59:59.925Z"
PROTOCOLS_PATH = "protocols/kuru_addresses.json"

DAY_STEM = START_UTC[:10]
CHECKPOINT_PATH = Path(f"kuru_blocks_{DAY_STEM}.checkpoint.json")
SUMMARY_PATH = Path(f"kuru_blocks_{DAY_STEM}.json")
BLOCKS_LIST_PATH = Path(f"kuru_blocks_{DAY_STEM}_blocks.json")

HYPERSYNC_TIMEOUT_MS = 100_000  # 100s per request (default 30s)
HYPERSYNC_MAX_NUM_BLOCKS = 2_000
HYPERSYNC_MAX_NUM_TRACES = 50_000
# ===================================

load_dotenv()
HYPERSYNC_URL = os.environ.get("HYPERSYNC_URL", "").strip()
RPC_URL = os.environ.get("MONAD_RPC_URL", "").strip()
ENVIO_API_TOKEN = os.environ.get("ENVIO_API_TOKEN", "").strip()
if not HYPERSYNC_URL or not RPC_URL or not ENVIO_API_TOKEN:
    raise RuntimeError("Set private RPC/indexer environment variables before running this notebook")

print("Private data sources configured")
print(f"UTC window: {START_UTC} .. {END_UTC}")


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Private data sources configured
UTC window: 2026-06-25T00:00:00.000Z .. 2026-06-25T23:59:59.925Z


In [4]:
ROLE_SUFFIXES = ("Impl", "Operator", "BackrunOperator", "FastlaneEntryPoint")
ACTIVE_VAULT_ROLES = ("Operator", "Market", "Vault", "Impl")


def _strip_role_suffix(rest: str, roles: tuple[str, ...]) -> str:
    for role in roles:
        suffix = f"-{role}"
        if rest.endswith(suffix):
            return rest[: -len(suffix)]
    return rest


def market_group_key(contract_name: str) -> Optional[str]:
    if contract_name.startswith("Market-"):
        rest = _strip_role_suffix(contract_name[len("Market-") :], ROLE_SUFFIXES)
        return rest.replace("-", "/")
    if contract_name.startswith("ActiveVault-"):
        rest = _strip_role_suffix(contract_name[len("ActiveVault-") :], ACTIVE_VAULT_ROLES)
        return f"ActiveVault/{rest.replace('-', '/')}"
    return None


def is_primary_market_contract(contract_name: str) -> bool:
    return contract_name.startswith("Market-") and not any(
        contract_name.endswith(f"-{role}") for role in ROLE_SUFFIXES
    )


def load_kuru_addresses(path: str) -> tuple[list[str], dict[str, str], dict[str, list[str]], dict[str, str]]:
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    addrs: list[str] = []
    labels: dict[str, str] = {}
    groups: dict[str, list[str]] = {"Core": [], "Additional": []}
    primary_market: dict[str, str] = {}
    seen: set[str] = set()

    for proto in data.get("protocols", []):
        if proto.get("name") != "Kuru":
            continue
        for name, addr in (proto.get("contracts") or {}).items():
            a = str(addr).strip().lower()
            if not a.startswith("0x") or len(a) != 42:
                raise ValueError(f"invalid address for {name}: {addr!r}")

            group = market_group_key(name)
            if group:
                groups.setdefault(group, []).append(a)
                if is_primary_market_contract(name) or name.endswith("-Market"):
                    primary_market[group] = a
            elif name.startswith("Additional-"):
                groups["Additional"].append(a)
            else:
                groups["Core"].append(a)

            if a not in seen:
                addrs.append(a)
                seen.add(a)
            labels.setdefault(a, str(name))

    groups = {k: v for k, v in groups.items() if v}
    for group, group_addrs in groups.items():
        primary_market.setdefault(group, group_addrs[0])

    if not addrs:
        raise ValueError(f"no Kuru contracts in {path}")
    return addrs, labels, groups, primary_market


kuru_addresses, kuru_labels, market_groups, market_primary = load_kuru_addresses(PROTOCOLS_PATH)
kuru_address_set = set(kuru_addresses)
addr_to_group = {addr: group for group, addrs in market_groups.items() for addr in addrs}

print(f"Loaded {len(kuru_addresses)} unique addresses across {len(market_groups)} groups:")
for group, addrs in sorted(market_groups.items()):
    print(f"  {group:<28} {len(addrs)} addrs  primary={market_primary[group]}")

Loaded 44 unique addresses across 17 groups:
  AUSD/USDC                    1 addrs  primary=0x699abc15308156e9a3ab89ec7387e9cfe1c86a3b
  AUSD/USDC2                   1 addrs  primary=0x8cf49e35d73b19433ff4d4421637aabb680dc9cc
  ActiveVault/MON/USDC         4 addrs  primary=0x065c9d28e428a0db40191a54d33d5b7c71a9c394
  Additional                   2 addrs  primary=0x742411e47c4f35d733138640d8be6047754b36c5
  CETES/USDC                   1 addrs  primary=0x52ceb71a33f5f92f030d3a74ceaa84e7d95c74ce
  CHOG/USDC                    1 addrs  primary=0x5e5166f02b8f91ab80833270435172078f4178ca
  Core                         12 addrs  primary=0xb3e6778480b2e488385e8205ea05e20060b813cb
  KUFA/MON                     1 addrs  primary=0x698bc39ca67b02d25688f40f16fd72d4a4b191f2
  MON/AUSD                     1 addrs  primary=0xf39c4fd5465ea2dd7b0756cebc48a258b34febf3
  MON/USDC                     5 addrs  primary=0xd0f8a6422ccdd812f29d8fb75cf5fcd41483badc
  SHMON/MON                    1 addrs  prim

In [5]:
def parse_utc(ts: str) -> int:
    ts = ts.strip().replace("Z", "+00:00")
    dt = datetime.fromisoformat(ts)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return int(dt.timestamp())


def rpc_call(method: str, params: list, timeout: int = 60) -> dict:
    resp = requests.post(
        RPC_URL,
        json={"jsonrpc": "2.0", "method": method, "params": params, "id": 1},
        timeout=timeout,
    )
    resp.raise_for_status()
    body = resp.json()
    if body.get("error"):
        raise RuntimeError(body["error"])
    return body["result"]


def get_latest_block() -> int:
    return int(rpc_call("eth_blockNumber", []), 16)


def get_block_timestamp(block_num: int) -> Optional[int]:
    block = rpc_call("eth_getBlockByNumber", [hex(block_num), False])
    if not block:
        return None
    return int(block["timestamp"], 16)


def find_block_at_or_after(target_ts: int, latest_block: int) -> Optional[int]:
    lo, hi, res = 1, latest_block, None
    while lo <= hi:
        mid = (lo + hi) // 2
        ts = get_block_timestamp(mid)
        if ts is None:
            lo = mid + 1
            continue
        if ts < target_ts:
            lo = mid + 1
        else:
            res = mid
            hi = mid - 1
    return res


def find_block_at_or_before(target_ts: int, latest_block: int, lo_bound: int = 0) -> Optional[int]:
    lo, hi, res = max(0, lo_bound), latest_block, None
    while lo <= hi:
        mid = (lo + hi) // 2
        ts = get_block_timestamp(mid)
        if ts is None:
            lo = mid + 1
            continue
        if ts > target_ts:
            hi = mid - 1
        else:
            res = mid
            lo = mid + 1
    return res


start_ts = parse_utc(START_UTC)
end_ts = parse_utc(END_UTC)
latest = get_latest_block()

print(f"Resolving UTC window to blocks (latest={latest:,})...")
start_block = find_block_at_or_after(start_ts, latest)
if start_block is None:
    raise RuntimeError("could not resolve start block")
end_block = find_block_at_or_before(end_ts, latest, lo_bound=start_block)
if end_block is None:
    raise RuntimeError("could not resolve end block")

total_blocks_in_window = end_block - start_block + 1
print(f"Blocks: {start_block:,} .. {end_block:,} ({total_blocks_in_window:,} blocks)")

Resolving UTC window to blocks (latest=83,822,902)...
Blocks: 83,488,158 .. 83,703,985 (215,828 blocks)


In [6]:
def _blocks_by_addr_from_checkpoint(raw: Optional[dict]) -> dict[str, set[int]]:
    out: dict[str, set[int]] = {}
    for addr, blocks in (raw or {}).items():
        out[str(addr).lower()] = set(int(b) for b in blocks)
    return out


def _serialize_blocks_by_addr(blocks_by_addr: dict[str, set[int]]) -> dict[str, list[int]]:
    return {addr: sorted(blocks) for addr, blocks in blocks_by_addr.items() if blocks}


def load_checkpoint() -> dict:
    if not CHECKPOINT_PATH.is_file():
        return {
            "phase": "traces",
            "trace_from_block": start_block,
            "tx_from_block": start_block,
            "trace_blocks": [],
            "tx_blocks": [],
            "trace_to_counts": {},
            "tx_to_counts": {},
            "trace_blocks_by_addr": {},
            "tx_blocks_by_addr": {},
        }
    data = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    data.setdefault("trace_to_counts", {})
    data.setdefault("tx_to_counts", {})
    data.setdefault("trace_blocks_by_addr", {})
    data.setdefault("tx_blocks_by_addr", {})
    return data


def save_checkpoint(state: dict) -> None:
    tmp = CHECKPOINT_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=2), encoding="utf-8")
    tmp.replace(CHECKPOINT_PATH)


def bump_to_count(counter: dict[str, int], addr: Optional[str]) -> None:
    if not addr:
        return
    key = addr.lower()
    counter[key] = counter.get(key, 0) + 1


async def collect_blocks_from_query(
    client: hypersync.HypersyncClient,
    query: hypersync.Query,
    end_block_exclusive: int,
    mode: str,
    blocks_out: set[int],
    blocks_by_addr: dict[str, set[int]],
    to_counts: dict[str, int],
    desc: str,
    checkpoint_fn=None,
) -> None:
    pbar = tqdm(total=end_block_exclusive - query.from_block, desc=desc, unit="blk")
    last_from = query.from_block
    try:
        while query.from_block < end_block_exclusive:
            res = await client.get(query)
            rows = res.data.traces if mode == "traces" else res.data.transactions
            for row in rows or []:
                bn = row.block_number
                to_addr = (row.to or "").lower()
                if bn is not None and start_block <= bn <= end_block:
                    blocks_out.add(bn)
                    if to_addr in kuru_address_set:
                        blocks_by_addr.setdefault(to_addr, set()).add(bn)
                bump_to_count(to_counts, row.to)

            advanced = max(0, res.next_block - query.from_block)
            pbar.update(advanced)
            query.from_block = res.next_block

            if checkpoint_fn is not None:
                checkpoint_fn(query.from_block)

            if res.next_block <= last_from:
                break
            last_from = res.next_block
    finally:
        pbar.close()


def build_trace_query(from_block: int) -> hypersync.Query:
    # Same shape as a single-address trace query, but `to` = all Kuru contracts.
    # Optional sighash filter (see reference cell below) not used here — we want
    # every internal/top-level call to any Kuru address, not one function selector.
    return hypersync.Query(
        from_block=from_block,
        to_block=end_block + 1,
        traces=[hypersync.TraceSelection(to=kuru_addresses)],
        field_selection=hypersync.FieldSelection(
            trace=[
                hypersync.TraceField.TO,
                hypersync.TraceField.BLOCK_NUMBER,
            ],
        ),
        join_mode=hypersync.JoinMode.JOIN_NOTHING,
        max_num_blocks=HYPERSYNC_MAX_NUM_BLOCKS,
        max_num_traces=HYPERSYNC_MAX_NUM_TRACES,
    )


def build_tx_query(from_block: int) -> hypersync.Query:
    return hypersync.Query(
        from_block=from_block,
        to_block=end_block + 1,
        transactions=[hypersync.TransactionSelection(to=kuru_addresses)],
        field_selection=hypersync.FieldSelection(
            transaction=[
                hypersync.TransactionField.BLOCK_NUMBER,
                hypersync.TransactionField.TO,
            ],
        ),
        join_mode=hypersync.JoinMode.JOIN_NOTHING,
        max_num_blocks=HYPERSYNC_MAX_NUM_BLOCKS,
        max_num_transactions=HYPERSYNC_MAX_NUM_TRACES,
    )


async def run_hypersync_scan() -> dict:
    client = hypersync.HypersyncClient(
        hypersync.ClientConfig(
            url=HYPERSYNC_URL,
            api_token=ENVIO_API_TOKEN,
            http_req_timeout_millis=HYPERSYNC_TIMEOUT_MS,
            max_num_retries=12,
        )
    )
    state = load_checkpoint()
    trace_blocks = set(state["trace_blocks"])
    tx_blocks = set(state["tx_blocks"])
    trace_to_counts = {k: int(v) for k, v in state["trace_to_counts"].items()}
    tx_to_counts = {k: int(v) for k, v in state["tx_to_counts"].items()}
    trace_blocks_by_addr = _blocks_by_addr_from_checkpoint(state.get("trace_blocks_by_addr"))
    tx_blocks_by_addr = _blocks_by_addr_from_checkpoint(state.get("tx_blocks_by_addr"))

    if state["phase"] == "done" and not state.get("trace_blocks_by_addr"):
        print(
            "WARN: checkpoint lacks per-address block data. "
            f"Delete {CHECKPOINT_PATH.name} and re-run for market activity breakdown."
        )

    end_exclusive = end_block + 1
    t0 = time.time()

    def save_trace_progress(from_block: int) -> None:
        state.update(
            {
                "phase": "traces",
                "trace_from_block": from_block,
                "trace_blocks": sorted(trace_blocks),
                "trace_to_counts": trace_to_counts,
                "trace_blocks_by_addr": _serialize_blocks_by_addr(trace_blocks_by_addr),
            }
        )
        save_checkpoint(state)

    def save_tx_progress(from_block: int) -> None:
        state.update(
            {
                "phase": "transactions",
                "tx_from_block": from_block,
                "tx_blocks": sorted(tx_blocks),
                "tx_to_counts": tx_to_counts,
                "tx_blocks_by_addr": _serialize_blocks_by_addr(tx_blocks_by_addr),
                "trace_from_block": end_exclusive,
                "trace_blocks": sorted(trace_blocks),
                "trace_to_counts": trace_to_counts,
                "trace_blocks_by_addr": _serialize_blocks_by_addr(trace_blocks_by_addr),
            }
        )
        save_checkpoint(state)

    if state["phase"] == "done":
        print("Checkpoint phase=done — using saved trace/tx block sets")
    elif state["phase"] == "traces":
        print(
            f"Phase 1/2: traces from block {state['trace_from_block']:,} "
            f"(timeout={HYPERSYNC_TIMEOUT_MS // 1000}s, max_blocks={HYPERSYNC_MAX_NUM_BLOCKS})"
        )
        trace_query = build_trace_query(state["trace_from_block"])
        await collect_blocks_from_query(
            client,
            trace_query,
            end_exclusive,
            "traces",
            trace_blocks,
            trace_blocks_by_addr,
            trace_to_counts,
            "HyperSync traces",
            checkpoint_fn=save_trace_progress,
        )
        state.update(
            {
                "phase": "transactions",
                "trace_from_block": end_exclusive,
                "trace_blocks": sorted(trace_blocks),
                "trace_to_counts": trace_to_counts,
                "trace_blocks_by_addr": _serialize_blocks_by_addr(trace_blocks_by_addr),
            }
        )
        save_checkpoint(state)

    if state["phase"] == "transactions":
        print("Phase 2/2: top-level transactions to Kuru")
        tx_query = build_tx_query(state["tx_from_block"])
        await collect_blocks_from_query(
            client,
            tx_query,
            end_exclusive,
            "transactions",
            tx_blocks,
            tx_blocks_by_addr,
            tx_to_counts,
            "HyperSync txs",
            checkpoint_fn=save_tx_progress,
        )
        state.update(
            {
                "phase": "done",
                "tx_from_block": end_exclusive,
                "tx_blocks": sorted(tx_blocks),
                "tx_to_counts": tx_to_counts,
                "tx_blocks_by_addr": _serialize_blocks_by_addr(tx_blocks_by_addr),
            }
        )
        save_checkpoint(state)

    elapsed = time.time() - t0
    both = trace_blocks & tx_blocks
    union = trace_blocks | tx_blocks

    return {
        "trace_blocks": trace_blocks,
        "tx_blocks": tx_blocks,
        "both_blocks": both,
        "union_blocks": union,
        "trace_to_counts": trace_to_counts,
        "tx_to_counts": tx_to_counts,
        "trace_blocks_by_addr": trace_blocks_by_addr,
        "tx_blocks_by_addr": tx_blocks_by_addr,
        "elapsed_sec": elapsed,
    }

### HyperSync reference query (single address + sighash)

Use this pattern for ad-hoc trace lookups. The main scan above uses the same API but filters `to` on **all** Kuru addresses and omits `sighash` so every call counts.

```python
query = hypersync.Query(
    from_block=83_800_000,
    to_block=83_806_022,  # exclusive
    traces=[hypersync.TraceSelection(
        to=["0xd0f8a6422ccdd812f29d8fb75cf5fcd41483badc"],
        sighash=["0x........"],  # e.g. cancelAllReplace selector
    )],
    field_selection=hypersync.FieldSelection(
        trace=[
            hypersync.TraceField.FROM,
            hypersync.TraceField.TO,
            hypersync.TraceField.SIGHASH,
            hypersync.TraceField.BLOCK_NUMBER,
            hypersync.TraceField.TRANSACTION_HASH,
        ],
        transaction=[
            hypersync.TransactionField.HASH,
            hypersync.TransactionField.FROM,
            hypersync.TransactionField.TO,
        ],
    ),
    # join_mode omitted => Default: traces also pull parent tx + block
)
res = await client.get(query)
```

**vs block-count scan:** we use `JoinNothing` and only trace fields to keep responses small over ~215k blocks.

In [7]:
def build_market_activity(scan: dict) -> list[dict]:
    trace_by = scan["trace_blocks_by_addr"]
    tx_by = scan["tx_blocks_by_addr"]
    trace_counts = scan["trace_to_counts"]
    tx_counts = scan["tx_to_counts"]
    rows: list[dict] = []

    for group, addrs in market_groups.items():
        trace_blocks_g: set[int] = set()
        tx_blocks_g: set[int] = set()
        trace_hits = 0
        tx_hits = 0
        for addr in addrs:
            trace_blocks_g |= trace_by.get(addr, set())
            tx_blocks_g |= tx_by.get(addr, set())
            trace_hits += trace_counts.get(addr, 0)
            tx_hits += tx_counts.get(addr, 0)

        union_g = trace_blocks_g | tx_blocks_g
        primary = market_primary[group]
        primary_union = trace_by.get(primary, set()) | tx_by.get(primary, set())

        rows.append(
            {
                "group": group,
                "primary_market": primary,
                "address_count": len(addrs),
                "active_blocks": len(union_g),
                "primary_market_active_blocks": len(primary_union),
                "pct_of_window": round(100.0 * len(union_g) / total_blocks_in_window, 4)
                if total_blocks_in_window
                else 0.0,
                "trace_hits": trace_hits,
                "tx_hits": tx_hits,
                "total_hits": trace_hits + tx_hits,
                "trace_only_blocks": len(trace_blocks_g - tx_blocks_g),
                "tx_only_blocks": len(tx_blocks_g - trace_blocks_g),
                "both_blocks": len(trace_blocks_g & tx_blocks_g),
            }
        )

    return sorted(rows, key=lambda r: (-r["active_blocks"], -r["total_hits"], r["group"]))


scan = await run_hypersync_scan()

trace_blocks = scan["trace_blocks"]
tx_blocks = scan["tx_blocks"]
both_blocks = scan["both_blocks"]
union_blocks = scan["union_blocks"]
market_activity = build_market_activity(scan)

blocks_with_kuru = len(union_blocks)
pct_active = 100.0 * blocks_with_kuru / total_blocks_in_window if total_blocks_in_window else 0.0

print("\n" + "=" * 90)
print("KURU ACTIVE BLOCKS")
print("=" * 90)
print(f"UTC: {START_UTC} .. {END_UTC}")
print(f"Blocks: {start_block:,} .. {end_block:,} ({total_blocks_in_window:,} total)")
print(f"Blocks with Kuru activity: {blocks_with_kuru:,} ({pct_active:.2f}%)")
print(f"  via traces only:         {len(trace_blocks - tx_blocks):,}")
print(f"  via top-level tx only:   {len(tx_blocks - trace_blocks):,}")
print(f"  via both:                {len(both_blocks):,}")
print(f"HyperSync scan time:       {scan['elapsed_sec']:.1f}s")
print("=" * 90)

print("\nMARKET / GROUP ACTIVITY (sorted by active blocks)")
print(f"{'Group':<24} {'ActiveBlk':>10} {'%Window':>8} {'PrimaryBlk':>11} {'TraceHit':>10} {'TxHit':>10} {'TotalHit':>10}")
print("-" * 90)
for row in market_activity:
    print(
        f"{row['group']:<24} {row['active_blocks']:>10,} {row['pct_of_window']:>7.2f}% "
        f"{row['primary_market_active_blocks']:>11,} {row['trace_hits']:>10,} "
        f"{row['tx_hits']:>10,} {row['total_hits']:>10,}"
    )
print("=" * 90)

Phase 1/2: traces (internal + top-level calls to Kuru)


HyperSync traces:  43%|████▎     | 93426/215828 [03:34<1:08:43, 29.69blk/s][2026-06-26T13:18:13Z ERROR hypersync_client] failed to get arrow data from server, retrying... The error was: read response body bytes
    
    Caused by:
        0: error decoding response body
        1: request or response body error
        2: operation timed out
[2026-06-26T13:18:43Z ERROR hypersync_client] failed to get arrow data from server, retrying... The error was: read response body bytes
    
    Caused by:
        0: error decoding response body
        1: request or response body error
        2: operation timed out
HyperSync traces:  45%|████▌     | 97125/215828 [05:50<49:52, 39.66blk/s]  [2026-06-26T13:20:18Z ERROR hypersync_client] failed to get arrow data from server, retrying... The error was: read response body bytes
    
    Caused by:
        0: error decoding response body
        1: request or response body error
        2: operation timed out
[2026-06-26T13:20:49Z ERROR hypersync_clien

Phase 2/2: top-level transactions to Kuru


HyperSync txs: 100%|██████████| 215828/215828 [00:01<00:00, 116722.91blk/s]


KURU ACTIVE BLOCKS
UTC: 2026-06-25T00:00:00.000Z .. 2026-06-25T23:59:59.925Z
Blocks: 83,488,158 .. 83,703,985 (215,828 total)
Blocks with Kuru activity: 186,044 (86.20%)
  via traces only:         163,142
  via top-level tx only:   0
  via both:                22,902
HyperSync scan time:       1935.0s

MARKET / GROUP ACTIVITY (sorted by active blocks)
Group                     ActiveBlk  %Window  PrimaryBlk   TraceHit      TxHit   TotalHit
------------------------------------------------------------------------------------------
Core                        185,998   86.18%         130  9,714,308        579  9,714,887
ActiveVault/MON/USDC         99,802   46.24%      99,801  1,851,790     31,400  1,883,190
MON/USDC                     83,106   38.51%      22,570    168,435          4    168,439
WETH/USDC                    83,106   38.51%       8,791    140,782          3    140,785
cbBTC/USDC                   83,105   38.51%      57,152    237,504         12    237,516
XAUt0/USDC    

In [ ]:
def counts_by_contract(counter: dict[str, int]) -> list[dict]:
    rows = []
    for addr, count in sorted(counter.items(), key=lambda x: (-x[1], x[0])):
        rows.append(
            {
                "group": addr_to_group.get(addr, "?"),
                "contract_name": kuru_labels.get(addr, "?"),
                "address": addr,
                "count": count,
            }
        )
    return rows


summary = {
    "utc_window": {"start": START_UTC, "end": END_UTC},
    "block_range": {
        "start": start_block,
        "end": end_block,
        "count": total_blocks_in_window,
    },
    "blocks_with_kuru_activity": blocks_with_kuru,
    "pct_blocks_active": round(pct_active, 4),
    "overlap": {
        "trace_only_blocks": len(trace_blocks - tx_blocks),
        "tx_only_blocks": len(tx_blocks - trace_blocks),
        "both_blocks": len(both_blocks),
    },
    "market_activity": market_activity,
    "trace_row_counts_by_to": counts_by_contract(scan["trace_to_counts"]),
    "tx_row_counts_by_to": counts_by_contract(scan["tx_to_counts"]),
    "hypersync_url": HYPERSYNC_URL,
    "scan_elapsed_sec": round(scan["elapsed_sec"], 2),
}

MARKET_ACTIVITY_PATH = Path(f"kuru_blocks_{DAY_STEM}_markets.json")
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
MARKET_ACTIVITY_PATH.write_text(json.dumps(market_activity, indent=2), encoding="utf-8")
BLOCKS_LIST_PATH.write_text(
    json.dumps(sorted(union_blocks), indent=2),
    encoding="utf-8",
)

print(f"Saved summary -> {SUMMARY_PATH}")
print(f"Saved market activity -> {MARKET_ACTIVITY_PATH}")
print(f"Saved block list -> {BLOCKS_LIST_PATH} ({blocks_with_kuru:,} blocks)")

## Optional sanity check

If you also ran `count_protocol_txs_window.py` for the same day, compare unique blocks from its `log_touch` hash stores vs this notebook's trace-based block set. HyperSync trace coverage should be >= log-only block coverage.

In [ ]:
# Re-run only if you have log_touch sidecar hashes from count_protocol_txs_window.py
LOG_HASHES_DIR = Path(f"protocol_tx_counts_{DAY_STEM}.hashes/log_touch")

if LOG_HASHES_DIR.is_dir():
    print("Note: log_touch sidecars store tx hashes, not block numbers.")
    print("For a direct block-level comparison, re-fetch block numbers for those txs via RPC.")
    print(f"Found log_touch dir: {LOG_HASHES_DIR}")
else:
    print(f"No log_touch sidecar at {LOG_HASHES_DIR} — skip comparison.")